In [1]:
%pip install langgraph langchain langchain-community transformers accelerate datasets rank_bm25 duckduckgo-search

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from transformers import pipeline

In [5]:
generator = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=50
)

print("Modelo cargado")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Modelo cargado


In [4]:
from langchain_core.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun

In [5]:
search_tool = DuckDuckGoSearchRun()

In [6]:
def calculator(expression: str):
    
    try:
        return str(eval(expression))
    
    except Exception:
        return "Calculation error"

In [7]:
calculator_tool = Tool(
    name="calculator",
    func=calculator,
    description="Performs mathematical calculations."
)

In [8]:
def weather_tool(city: str):
    
    return f"The weather in {city} is clear."

In [9]:
weather = Tool(
    name="weather_tool",
    func=weather_tool,
    description="Returns weather information."
)

In [10]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

In [11]:
class AgentState(TypedDict):
    question: str
    answer: str

In [12]:
def agent_node(state):

    question = state["question"].lower()

    if "weather" in question:

        answer = weather.invoke("Paris")

    elif any(char.isdigit() for char in question):

        expression = (
            question.replace("what is", "")
            .replace("?", "")
            .strip()
        )

        answer = calculator_tool.invoke(expression)

    else:

        try:
            answer = search_tool.invoke(question)

        except Exception:
            answer = "Search failed."

    return {
        "answer": answer
    }

In [13]:
builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)

builder.add_edge(START, "agent")
builder.add_edge("agent", END)

graph = builder.compile()

In [14]:
result = graph.invoke({
    "question": "What is 25 * 12?",
    "answer": ""
})

print(result)

{'question': 'What is 25 * 12?', 'answer': '300'}


In [15]:
result = graph.invoke({
    "question": "What is the weather today?",
    "answer": ""
})

print(result)

{'question': 'What is the weather today?', 'answer': 'The weather in Paris is clear.'}


In [16]:
result = graph.invoke({
    "question": "Who is Marie Curie?",
    "answer": ""
})

print(result)

{'question': 'Who is Marie Curie?', 'answer': '2 weeks ago - Maria Salomea Skłodowska Curie (Polish: [ˈmarja salɔˈmɛa skwɔˈdɔfska kiˈri] ⓘ; née Skłodowska; 7 November 1867 – 4 July 1934), better known as Marie Curie (/ˈkjʊəri/ KURE-ee; French: [maʁi kyʁi] ⓘ), was a Polish and naturalised-French physicist and chemist. March 25, 2026 - Maria Salomea Skłodowska–Curie (7 November 1867 – 4 July 1934) was a Polish physicist and chemist. She did research on radioactivity. She was also the first woman to win a Nobel Prize. March 20, 2026 - Marie Curie, a Polish-born French physicist, is famous for her work on radioactivity. She was the first woman to win a Nobel Prize, and she is the only woman to win the award in two different fields (Physics, 1903; Chemistry, 1911). August 20, 2025 - She was and still is the only person to be awarded Nobel Prizes in two scientific categories. By that time, Curie was world-famous, and the director of the Curie Laboratory at the newly established Radium Instit